# clue_agents

> Channels that carry clues (blackboard, messages to two teammates, a routing coordinator), what each agent knows, and the numpy rule agents (Phase 11): share a needed clue, open the most-known subtask, submit when determined, guess at a set rate; and a scripted liar posting false clues.

In [ ]:
#| default_exp clue_agents

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from collections import defaultdict

from explRL.multi import Outcome
from explRL.team_agents import PatchMember


class ClueBoard:
    "Blackboard of clues: every shared clue per subtask, visible to all, with its author and time."
    blocked = frozenset()

    def __init__(self):
        self.posts, self.asks, self.stats = defaultdict(list), {}, defaultdict(lambda: dict(written=0, sent=0, received=0))

    def post(self, i, k, clue, t):
        "clue = (j, p, r), true or false (a liar's)."
        self.posts[k].append((*clue, i, float(t)))
        self.stats[i]['written'] += 1

    def view(self, i, t, board):
        "{k: [(j, p, r, author, t)]} for the open subtasks, newest first."
        return {k: sorted(self.posts[k], key=lambda x: -x[4]) for k in board.open if self.posts.get(k)}

    def ask(self, i, k, t):
        "Agent i's request (k, i, t) for clues of its open subtask k (Phase 13); replaces its earlier request."
        self.asks[i] = (k, i, float(t))

    def requests(self, board):
        "[(k, i, t)] still live: k open and agent i has opened nothing since asking (ordered by agent)."
        return [x for i, x in sorted(self.asks.items()) if x[0] in board.open
                and not any(r[0] == i and r[3] == 'travel' and r[1] > x[2] for r in board.ledger)]


class ClueMessages:
    "A share goes to k random teammates (seeded per sender); received clues are kept."
    def __init__(self, n, k=2, rngs=None):
        self.k, self.rngs = int(k), rngs
        self.inbox, self.seen = [[] for _ in range(n)], [[] for _ in range(n)]
        self.stats = defaultdict(lambda: dict(written=0, sent=0, received=0))

    def post(self, i, k, clue, t):
        others = [a for a in range(len(self.inbox)) if a != i]
        if not others:
            return []
        to = self.rngs[i].choice(others, min(self.k, len(others)), replace=False)
        for a in to:
            self.inbox[a].append((i, k, clue, float(t)))
            self.seen[a].append((i, k, clue, float(t)))
            self.stats[a]['received'] += 1
        self.stats[i]['sent'] += len(to)
        return [int(a) for a in to]

    def take(self, i):
        "Messages received since the last call."
        out, self.inbox[i] = self.inbox[i], []
        return out


class ClueLead:
    "Numpy coordinator: routes each shared clue to every agent with that subtask open; suggests subtasks."
    def __init__(self, n, passPooled=False):
        "passPooled: with each suggestion, also pass the suggested subtask's pooled clues (Track A, A2)."
        self.routed, self.pooled, self.author = [[] for _ in range(n)], defaultdict(dict), defaultdict(dict)
        self.passPooled = bool(passPooled)
        self.assigned, self.stats = [None] * n, defaultdict(lambda: dict(written=0, sent=0, received=0))

    def post(self, i, k, clue, t, board):
        self.pooled[k][clue[0]], self.author[k][clue[0]] = clue[2], i
        self.stats[i]['written'] += 1
        for a, o in enumerate(board.opened):
            if a != i and o == k:
                self.routed[a].append((i, k, clue, float(t)))
                self.stats[a]['received'] += 1

    def suggest(self, i, board):
        "The open subtask with the most pooled clues not assigned to another agent (ties: value, then lower id)."
        taken = {x for a, x in enumerate(self.assigned) if a != i}
        free = [k for k in board.open if k not in taken] or list(board.open)
        k = max(free, key=lambda k: (len(self.pooled.get(k, {})), board.tasks[k].value, -k))
        self.assigned[i] = k
        if self.passPooled:
            have = {(x[1], x[2][0]) for x in self.routed[i]}
            for j, r in sorted(self.pooled.get(k, {}).items()):
                if (k, j) not in have:
                    self.routed[i].append((self.author[k][j], k, (j, board.tasks[k].task.primes[j], r), board.tWorld))
                    self.stats[i]['received'] += 1
        return k


def known(i, board, channel, checked=None):
    """{k: {j: r}} for the open subtasks: clues agent i holds, plus those its channel showed it (own clues win).
    checked: agent i's checks {(k, j, r): true?} (Phase 13): clues found false are left out, those found true win."""
    seen = []
    if isinstance(channel, ClueBoard):
        seen = [(k, x[0], x[2]) for k, xs in channel.posts.items() for x in sorted(xs, key=lambda x: x[4])]
    elif isinstance(channel, ClueMessages):
        seen = [(k, c[0], c[2]) for _, k, c, _ in channel.seen[i]]
    elif isinstance(channel, ClueLead):
        seen = [(k, c[0], c[2]) for _, k, c, _ in channel.routed[i]]
    out, ck = {k: {} for k in board.open}, checked or {}
    for k, j, r in seen + [x for x, ok in ck.items() if ok]:   # later sightings overwrite earlier ones
        if k in out and ck.get((k, j, r)) is not False:
            out[k][j] = r
    for k in out:
        out[k].update(board.held(i, k))
    return out


def candidates(board, k, knownK):
    return board.tasks[k].task.candidates(knownK)


class RuleAgent(PatchMember):
    """Numpy clue-board agent: submit when determined; else share a known clue no one has shared (policy 'needed':
    for subtasks a teammate has open; 'always'; 'never'; never without a channel); else open the subtask with the
    most known clues; else guess with probability `guess`; else wait."""
    extraNames = ('target', 'harvest', 'nHere', 'patch')

    def __init__(self, channel, rng, share='needed', guess=0.0):
        self.channel, self.rng, self.share, self.guess = channel, rng, share, float(guess)

    def _shareable(self, b, kn, t):
        if self.channel is None or self.share == 'never':
            return None
        done = {(s[2], s[3]) for s in b.shareLog}
        others = {o for a, o in enumerate(b.opened) if a != self.id}
        for k in sorted(kn):
            if self.share == 'needed' and k not in others:
                continue
            for j, r in sorted(b.held(self.id, k).items()):
                if (k, j) not in done:
                    return ('share', (k, j, r))
        return None

    def choose(self, b, t):
        kn, k = known(self.id, b, self.channel), b.opened[self.id]
        if k in b.open and len(c := candidates(b, k, kn[k])) == 1:
            b.noteSubmit(self.id, 1)
            return ('submit', c[0])
        if act := self._shareable(b, kn, t):
            return act
        if isinstance(self.channel, ClueLead):
            top = [self.channel.suggest(self.id, b)]
        else:
            live = [x for x in b.open if b.tasks[x].expire > t + 1] or list(b.open)
            best = max(len(kn[x]) for x in live)
            top = [x for x in live if len(kn[x]) == best]
            top = [x for x in top if b.tasks[x].value == max(b.tasks[y].value for y in top)]
        if k not in top:
            return ('open', int(top[int(self.rng.integers(len(top)))]))
        if self.rng.random() < self.guess:
            c = candidates(b, k, kn[k])
            b.noteSubmit(self.id, len(c))
            return ('submit', int(c[int(self.rng.integers(len(c)))]))
        return ('wait', None)

    def post(self, b, action, t):
        if action[0] == 'share' and self.channel is not None:
            k, j, r = action[1]
            clue = (j, b.tasks[k].task.primes[j], r)
            if isinstance(self.channel, ClueLead):
                self.channel.post(self.id, k, clue, t, b)
            else:
                self.channel.post(self.id, k, clue, t)

    def event(self, world, t):
        world.advance(t)
        action = self.choose(world, t)
        reward, dt, loc, info = world.apply(self.id, action, t)
        self.post(world, action, t)
        return Outcome(dt, info['kind'], loc, reward, (action[1] if action[0] in ('open', 'submit') else -1,
                                                       info['harvest'], info['nHere'], info['patch']))


class ClueLiar(RuleAgent):
    "Scripted liar: on every other turn shares a false residue for a subtask it is not working on; else a rule agent."
    def __init__(self, channel, rng):
        super().__init__(channel, rng, share='never', guess=0.0)
        self.turn = 0

    def choose(self, b, t):
        self.turn += 1
        cand = [k for k in b.open if k != b.opened[self.id]]
        if self.channel is not None and self.turn % 2 and cand:
            k = int(cand[int(self.rng.integers(len(cand)))])
            task = b.tasks[k].task
            j = int(self.rng.integers(len(task.primes)))
            p, r = task.clue(j)
            return ('share', (k, j, int((r + 1 + self.rng.integers(p - 1)) % p)))
        return super().choose(b, t)

## Tests

In [ ]:
import numpy as np
from explRL.clue_agents import *
from explRL.clueboard import ClueBoardEnv
from explRL.multi import runTeam, spawnRngs

b = ClueBoardEnv(3, 4, m=3, r=1, R=105, life=100)
k = b.open[0]; t = b.tasks[k].task
# channels deliver shared clues: the board to all, messages to two, the coordinator to the openers
cb = ClueBoard(); cb.post(1, k, (0, t.primes[0], t.clue(0)[1]), 2.0)
assert cb.view(0, 3.0, b)[k][0][:3] == (0, t.primes[0], t.clue(0)[1]) and known(2, b, cb)[k][0] == t.clue(0)[1]
mb = ClueMessages(4, 2, spawnRngs(9, 4)); to = mb.post(1, k, (1, t.primes[1], t.clue(1)[1]), 2.0)
assert len(to) == 2 and 1 not in to and all(known(a, b, mb)[k].get(1) == t.clue(1)[1] for a in to)
assert len(mb.take(to[0])) == 1 and mb.take(to[0]) == [] and known(to[0], b, mb)[k][1] == t.clue(1)[1]   # kept after take
other = [a for a in range(4) if a not in to and a != 1][0]
assert 1 not in known(other, b, mb)[k] or 1 in b.held(other, k)
lead = ClueLead(4); b.opened[2] = k
lead.post(0, k, (2, t.primes[2], t.clue(2)[1]), 3.0, b)
assert known(2, b, lead)[k][2] == t.clue(2)[1] and (2 not in known(3, b, lead)[k] or 2 in b.held(3, k))
assert lead.suggest(0, b) == k and lead.assigned[0] == k
# own clues win over a false sighting
b.opened[2] = None
held = b.held(0, k)
if held:
    j, r = next(iter(held.items())); cb.post(1, k, (j, t.primes[j], (r + 1) % t.primes[j]), 4.0)
    assert known(0, b, cb)[k][j] == r

def team(channel, share='needed', seed=0, N=4, T=40, liar=False, **kw):
    bd = ClueBoardEnv(seed, N, life=30, **kw)
    ch = dict(none=None, blackboard=ClueBoard(), messages=ClueMessages(N, 2, spawnRngs(seed + 2000, N)), hierarchical=ClueLead(N))[channel]
    rngs = spawnRngs(seed + 1000, N)
    ms = [ClueLiar(ch, rngs[0]) if liar and i == 0 else RuleAgent(ch, rngs[i], share) for i in range(N)]
    runTeam(ms, bd, T)
    return bd
# AC-4: with no channel rule agents never share, and a channel with 'never' sharing gives the no-channel result
assert team('none').counts['shares'] == 0
assert team('blackboard', 'never').netByAgent().sum() == team('none').netByAgent().sum()
# the blackboard with 'needed' sharing solves more than no channel (averaged over seeds); assisted solves happen
bb = [team('blackboard', seed=s) for s in range(5)]; no = [team('none', seed=s) for s in range(5)]
assert sum(x.counts['solved'] for x in bb) > sum(x.counts['solved'] for x in no)
assert any(a for x in bb for (_, _, _, a) in x.solveLog)
for ch in ('messages', 'hierarchical'):
    assert team(ch).counts['shares'] > 0
# a solo agent holds every clue and solves without sharing
solo = team('none', N=1, T=30)
assert solo.counts['solved'] > 0 and solo.counts['wrong'] == 0
# a liar posts false clues; answers built on them can be wrong
lb = [team('blackboard', seed=s, liar=True) for s in range(5)]
assert all(any(x[6] for x in d.shareLog) for d in lb) and sum(d.counts['wrong'] for d in lb) > sum(d.counts['wrong'] for d in bb)
# guessing submits candidates and is reported to the board
g = ClueBoardEnv(1, 4)
ms = [RuleAgent(None, r, 'needed', guess=1.0) for r in spawnRngs(5, 4)]
runTeam(ms, g, 20)
assert g.counts['wrong'] > 0 and all(s[3] is not None and s[3] >= 1 for s in g.submits)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()